In [33]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

In [2]:
parsivel_1 = xr.open_dataset(r'..\data\Cabaw_parsivel\PAR001_Cabauw_202407.nc')
reference_date = np.datetime64("1970-01-01")

# Convert minutes into datetime
datetime_values = reference_date + parsivel_1.epoch_time.astype("timedelta64[s]")
parsivel_1['time'] = datetime_values


rain_rate = parsivel_1.rainfall_rate_32bit

In [13]:
rain_rate.time

<xarray.DataArray 'time' (time: 44641)> Size: 357kB
array(['2024-07-01T00:00:00', '2024-07-01T00:01:00', '2024-07-01T00:02:00',
       ..., '2024-07-31T23:58:00', '2024-07-31T23:59:00',
       '2024-08-01T00:00:00'], dtype='datetime64[s]')
Coordinates:
  * time     (time) datetime64[s] 357kB 2024-07-01 ... 2024-08-01
Attributes:
    long_name:  End time of observation period, in seconds since 1970-01-01 0...
    units:      s
    comment:    The length of a measurement interval is 1 minute. Values are ...

In [30]:
rain_std = rain_rate.rolling(time=10, center=True).std()
rain_std = rain_std.dropna(dim='time')
rain_class = xr.zeros_like(rain_std)

# Stratiform: std > 0 and < 1.5

# Initialize classification by no rain = 0 where std == 0, else NaN
rain_class = xr.where(rain_std == 0, 0, np.nan)

# Assign stratiform = 1 where 0 < std <= 1.5
rain_class = xr.where((rain_std > 0) & (rain_std <= 1.5), 1, rain_class)

# Assign convective = 2 where std > 1.5
rain_class = xr.where(rain_std > 1.5, 2, rain_class)

# Optionally convert to integer
rain_class = rain_class.astype(int)

In [35]:
mask = (rain_class == 2)

# Get the indexes of True values along the 'time' dimension
indexes = np.where(mask)[0]

print(indexes)

[ 1559  1560  1561  1562  1563  1564  1565  1566  1567  1568  1569  3303
  3304  3305  3306  3307  3308  3309  3310  3311  3312  3313  3343  3344
  3345  3346  3347  3348  3349  3350  3351  3352  3363  3364  3365  3366
  3367  3368  3369  3370  3371  3372  3373  3374  4286  4287  4288  4315
  4316  4317  4318  4319  4320  4321  4322  4323  4324  4325  4326  4327
  4328  4329  4330  4331  4332  4333  4334  4335  4362  4398  4399  4400
  4401  4402  4403  4404  4405  4406  4407  4408  4409  4410  4411  4412
  4413  4414  4415  4416  4417  4418  4419  4456  4457  4458  4459  4460
  4461  4462  4463  4464  4465  4473  4474  4475  4476  4477  4478  4479
  4480  4481  4482  4483  4484  4485  4486  4487  4488  4489  4490  4491
  4492  4493  4494  4495  4496  8373  8374  8375  8376  8377  8378  8379
  8380  8381  8382  8383  8433  8434  8435  8436  8437  8438  8439  8440
  8441  8442  8443  8444  8445  8446  8447  8448  8449  8450  8451  8452
  8453  8454  8455  8456  8457  8458  8459  8639  8

In [37]:
rain_class.time[indexes]

<xarray.DataArray 'time' (time: 523)> Size: 4kB
array(['2024-07-02T02:04:00', '2024-07-02T02:05:00', '2024-07-02T02:06:00',
       ..., '2024-07-26T01:54:00', '2024-07-26T01:55:00',
       '2024-07-26T01:56:00'], dtype='datetime64[s]')
Coordinates:
  * time     (time) datetime64[s] 4kB 2024-07-02T02:04:00 ... 2024-07-26T01:5...